# GV Train 100h Inventory for CallWhisper-8k

Run this on a standard Colab CPU. It downloads or reuses `GV_Train_100h`, excludes the frozen 100-file benchmark IDs, probes every audio file with visible progress, and saves inspectable CSV/JSON artifacts to Drive. It does not train a model.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/call-whisper')
REPO_DIR = Path('/content/CallWhisper-8k')
os.chdir('/content')  # Never delete the directory Colab is currently inside.
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tqdm>=4.66', 'pandas>=2.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
print('Repo ready:', REPO_DIR)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)
print('ffprobe:', shutil.which('ffprobe'))

In [ ]:
import requests
from tqdm.auto import tqdm

URL = 'https://www.openslr.org/resources/118/GV_Train_100h.tar.gz'
ARCHIVE_DIR = DRIVE_DIR / 'saved_datasets'
ARCHIVE = ARCHIVE_DIR / 'GV_Train_100h.tar.gz'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

def download(url, destination):
    temporary = destination.with_suffix(destination.suffix + '.part')
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        with temporary.open('wb') as handle, tqdm(total=total, unit='B', unit_scale=True, desc=destination.name) as progress:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    progress.update(len(chunk))
    temporary.replace(destination)

if ARCHIVE.exists() and ARCHIVE.stat().st_size > 1_000_000_000:
    print('Reusing Drive archive:', ARCHIVE)
else:
    if ARCHIVE.exists(): ARCHIVE.unlink()
    download(URL, ARCHIVE)
print('Archive size GB:', round(ARCHIVE.stat().st_size / 1e9, 3))

In [ ]:
import tarfile

EXTRACT_ROOT = Path('/content/callwhisper_data')
EXPECTED = EXTRACT_ROOT / 'GV_Train_100h'

def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(archive, 'r:gz') as tar:
        members = tar.getmembers()
        for member in tqdm(members, desc='Validating archive'):
            if member.issym() or member.islnk():
                raise RuntimeError(f'Refusing linked member: {member.name}')
            target = (destination / member.name).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f'Unsafe path: {member.name}')
        tar.extractall(destination, filter='data')

if not (EXPECTED / 'Audio').is_dir():
    if EXTRACT_ROOT.exists(): shutil.rmtree(EXTRACT_ROOT)
    safe_extract(ARCHIVE, EXTRACT_ROOT)
matches = [path for path in EXTRACT_ROOT.rglob('GV_Train_100h') if (path / 'Audio').is_dir()]
if not matches: raise FileNotFoundError('GV_Train_100h/Audio not found after extraction')
GV_TRAIN = matches[0]
if not (GV_TRAIN / 'text').exists(): raise FileNotFoundError(GV_TRAIN / 'text')
if not ((GV_TRAIN / 'mp3.scp').exists() or (GV_TRAIN / 'wav.scp').exists()): raise FileNotFoundError('Missing mp3.scp or wav.scp')
print('Dataset ready:', GV_TRAIN)
print('Audio files:', sum(path.is_file() for path in (GV_TRAIN / 'Audio').rglob('*')))

## Probe and Save

The progress bar below covers every indexed training file. Eight concurrent `ffprobe` workers keep this substantially faster than the old sequential notebook cell.

In [ ]:
OUTPUT_DIR = DRIVE_DIR / 'results/gv_train_100h_inventory'
FROZEN = REPO_DIR / 'datasets/manifests/gramvaani_dev_100.csv'
command = [sys.executable, '-m', 'callwhisper.datasets.gramvaani_inventory',
    '--dataset-dir', str(GV_TRAIN), '--frozen-manifest', str(FROZEN),
    '--output-dir', str(OUTPUT_DIR), '--min-duration-s', '1.0',
    '--max-duration-s', '30.0', '--workers', '8']
print('RUN:', ' '.join(command), flush=True)
subprocess.run(command, check=True, env={**os.environ, 'PYTHONPATH': str(REPO_DIR / 'src')})

In [ ]:
import json, pandas as pd
from IPython.display import display

summary = json.loads((OUTPUT_DIR / 'gv_train_100h_inventory_summary.json').read_text())
inventory = pd.read_csv(OUTPUT_DIR / 'gv_train_100h_inventory.csv')
rejected = pd.read_csv(OUTPUT_DIR / 'gv_train_100h_rejected.csv')
print(json.dumps(summary, indent=2, ensure_ascii=False))
display(inventory.groupby(['source_rate_group', 'sample_rate_hz']).size().rename('files').reset_index())
display(rejected['reason'].value_counts().rename_axis('reason').reset_index(name='files'))
frozen_ids = set(pd.read_csv(FROZEN)['audio_path'].map(lambda value: Path(value).stem))
assert not (set(inventory['utterance_id']) & frozen_ids), 'Frozen benchmark leakage detected'
print('Leakage check passed: 0 frozen IDs in inventory')
print('Saved permanently to:', OUTPUT_DIR)

## Stop Here

Do not train yet. Inspect the inventory summary, sample-rate groups, transcript markers, and rejection reasons. The next step is deterministic curation and train/internal-eval splitting.